In [1]:
!apt-get update -qq
!apt-get install -y flex bison gcc

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:11.2.0-1ubuntu1).
gcc set to manually installed.
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison-doc flex-doc
The following NEW packages will be installed:
  bison flex libfl-dev libfl2
0 upgraded, 4 newly installed, 0 to remove and 78 not upgraded.
Need to get 1,072 kB of archives.
After this operation, 3,667 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 flex amd64 2.6.4-8build2 [307 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 bison amd64 2:3.8.2+dfsg-1build1 [748 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl2 amd64 2.6.4-8build2 

In [2]:
%%writefile optimize.l
%{
#include "optimize.tab.h"
#include <string.h>
#include <stdlib.h>
%}

%%

[a-zA-Z][a-zA-Z0-9]* {
    yylval.str = strdup(yytext);
    return ID;
}

[0-9]+ {
    yylval.str = strdup(yytext);
    return NUM;
}

"=" {
    return '=';
}

"+" {
    return '+';
}

"-" {
    return '-';
}

"*" {
    return '*';
}

"/" {
    return '/';
}

";" {
    return ';';
}

[ \t\n]+ {
    /* skip whitespace */
}

. {
    return yytext[0];
}

%%

int yywrap()
{
    return 1;
}

Writing optimize.l


In [3]:
%%writefile optimize.y
%{
#include <stdio.h>
#include <string.h>
#include <stdlib.h>
#include <ctype.h>

int yylex(void);
int yyerror(char *s);
%}

%union {
    char *str;
}

%token <str> ID NUM
%type <str> expr

%left '+' '-'
%left '*' '/'

%%

stmt_list:
      stmt_list stmt
    | stmt
    ;

stmt:
      ID '=' expr ';'
      {
          printf("%s = %s\n", $1, $3);
      }
    ;

expr:
      NUM
      {
          $$ = $1;
      }

    | ID
      {
          $$ = $1;
      }

    | expr '+' expr
      {
          if (isdigit($1[0]) && isdigit($3[0]))
          {
              char buf[20];

              sprintf(buf, "%d", atoi($1) + atoi($3));

              $$ = strdup(buf);

              printf("// Constant Folding: %s + %s -> %s\n",
                     $1, $3, $$);
          }
          else if (strcmp($3, "0") == 0)
          {
              $$ = $1;

              printf("// Algebraic Simplification: x + 0 -> x\n");
          }
          else if (strcmp($1, "0") == 0)
          {
              $$ = $3;

              printf("// Algebraic Simplification: 0 + x -> x\n");
          }
          else
          {
              char buf[40];

              sprintf(buf, "%s + %s", $1, $3);

              $$ = strdup(buf);
          }
      }

    | expr '-' expr
      {
          if (isdigit($1[0]) && isdigit($3[0]))
          {
              char buf[20];

              sprintf(buf, "%d", atoi($1) - atoi($3));

              $$ = strdup(buf);

              printf("// Constant Folding: %s - %s -> %s\n",
                     $1, $3, $$);
          }
          else if (strcmp($3, "0") == 0)
          {
              $$ = $1;

              printf("// Algebraic Simplification: x - 0 -> x\n");
          }
          else
          {
              char buf[40];

              sprintf(buf, "%s - %s", $1, $3);

              $$ = strdup(buf);
          }
      }

    | expr '*' expr
      {
          if (isdigit($1[0]) && isdigit($3[0]))
          {
              char buf[20];

              sprintf(buf, "%d", atoi($1) * atoi($3));

              $$ = strdup(buf);

              printf("// Constant Folding: %s * %s -> %s\n",
                     $1, $3, $$);
          }
          else if (strcmp($3, "1") == 0)
          {
              $$ = $1;

              printf("// Algebraic Simplification: x * 1 -> x\n");
          }
          else if (strcmp($3, "2") == 0)
          {
              char buf[40];

              sprintf(buf, "%s + %s", $1, $1);

              $$ = strdup(buf);

              printf("// Strength Reduction: x * 2 -> x + x\n");
          }
          else
          {
              char buf[40];

              sprintf(buf, "%s * %s", $1, $3);

              $$ = strdup(buf);
          }
      }

    | expr '/' expr
      {
          if (isdigit($1[0]) && isdigit($3[0]))
          {
              char buf[20];

              sprintf(buf, "%d", atoi($1) / atoi($3));

              $$ = strdup(buf);

              printf("// Constant Folding: %s / %s -> %s\n",
                     $1, $3, $$);
          }
          else if (strcmp($3, "1") == 0)
          {
              $$ = $1;

              printf("// Algebraic Simplification: x / 1 -> x\n");
          }
          else
          {
              char buf[40];

              sprintf(buf, "%s / %s", $1, $3);

              $$ = strdup(buf);
          }
      }
    ;

%%

int main()
{
    printf("Enter Three Address Code statements:\n");

    yyparse();

    return 0;
}

int yyerror(char *s)
{
    printf("Syntax Error: %s\n", s);

    return 0;
}

Writing optimize.y


In [4]:
!rm -f optimize.tab.c optimize.tab.h lex.yy.c optimize

In [5]:
!bison -d optimize.y

In [6]:
!flex optimize.l

In [7]:
!gcc lex.yy.c optimize.tab.c -o optimize -lfl

In [8]:
!echo "a = 2 + 4;" | ./optimize

Enter Three Address Code statements:
// Constant Folding: 2 + 4 -> 6
a = 6


In [9]:
!echo "b = d * 1;" | ./optimize

Enter Three Address Code statements:
// Algebraic Simplification: x * 1 -> x
b = d


In [10]:
!echo "c = s * 2;" | ./optimize

Enter Three Address Code statements:
// Strength Reduction: x * 2 -> x + x
c = s + s


In [11]:
!printf "a = 2 + 4;\nb = d * 1;\nc = s * 2;\n" | ./optimize

Enter Three Address Code statements:
// Constant Folding: 2 + 4 -> 6
a = 6
// Algebraic Simplification: x * 1 -> x
b = d
// Strength Reduction: x * 2 -> x + x
c = s + s
